
# Modelo metiendo las estaciones de fin en el contexto

In [52]:
version = "3_1"

In [53]:
import matplotlib.pyplot as plt

def show_history(history, model_name: str):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10))  # 2 filas, 1 columna

    # Pérdida (loss)
    ax1.plot(history.history['loss'],     label='Training Loss',  color='green')
    ax1.plot(history.history['val_loss'], label='Validation Loss', color='blue')
    ax1.set_title('Loss evolution')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()

    # Precisión (accuracy)
    ax2.plot(history.history['accuracy'],     label='Training Accuracy',  color='green')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', color='blue')
    ax2.set_title('Accuracy evolution')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.legend()

    # Título global
    fig.suptitle(model_name, fontsize=16)

    # Ajustar márgenes
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [54]:
import pickle
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_rows", None)   # Muestra todas las filas
pd.set_option("display.max_columns", None)  # Muestra todas las columnas
pd.set_option("display.width", None)     # No corta la tabla en varias líneas
pd.set_option("display.max_colwidth", None)  # Muestra el contenido de celdas completo

In [67]:
with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)
    
with open("../../data/bicycles/df_bicycles_stations.pk1", "rb") as f:
    df_bikesStations = pickle.load(f)

In [56]:
df_model = df_data.drop(columns=[
    "ride_id",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "time_hms_ms",
    "member_casual"
])

In [57]:
df_model.head()

,year,month,day,event,temperature,wind_speed,precipitation,relative_humidity,snow_depth,rideable_type_classic_bike,rideable_type_docked_bike,rideable_type_electric_bike,member_casual_bool,day_type_Holiday,day_type_Normal,day_type_Weekend,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,start_station_idx,end_station_idx,hour_float,hour_sin,hour_cos,month_sin,month_cos,duration_min
0,2022,1,4,False,-5.47,3.100000,0.0,74.85,0.0,True,False,False,True,False,True,False,-2.348318,-0.573362,0.916001,-0.099704,-0.056148,1281,1896,4.641111,0.937383,0.348299,0.5,0.866025,13.333333
1,2022,1,4,False,-5.00,4.600000,0.0,80.00,0.0,True,False,False,True,False,True,False,-2.299739,0.115036,1.211601,-0.099704,-0.056148,1819,1645,11.446944,0.144284,-0.989536,0.5,0.866025,8.683333
2,2022,1,4,False,-5.00,4.841667,0.0,82.90,0.0,True,False,False,False,False,True,False,-2.299739,0.225945,1.378055,-0.099704,-0.056148,1250,1250,13.231944,-0.316960,-0.948439,0.5,0.866025,38.533333
3,2022,1,12,False,0.75,2.850000,0.0,76.25,0.0,False,False,True,True,False,True,False,-1.705422,-0.688095,0.996358,-0.099704,-0.056148,1896,1911,15.498333,-0.793088,-0.609108,0.5,0.866025,6.183333
4,2022,1,16,False,-9.83,1.885000,0.0,74.30,0.0,True,False,False,True,False,False,True,-2.798966,-1.130964,0.884432,-0.099704,-0.056148,1893,1646,8.467778,0.798460,-0.602047,0.5,0.866025,30.133333


In [58]:
df_bikesStations.head()

,name,capacity,lon,lat,short_name,station_id,rental_uris,region_id,address,station_idx
286,2112 W Peterson Ave,11,-87.683593,41.991178,CHI00611,a3af2216-a135-11e9-9cda-0a87ae2ba916,"{'android': 'https://chi.lft.to/lastmile_qr_scan', 'ios': 'https://chi.lft.to/lastmile_qr_scan'}",NaN,NaN,1701
105,21st St & Pulaski Rd,12,-87.725090,41.854370,CHI01823,1931696364106218226,"{'android': 'https://chi.lft.to/lastmile_qr_scan', 'ios': 'https://chi.lft.to/lastmile_qr_scan'}",NaN,NaN,938
88,63rd St Beach,15,-87.576324,41.780911,CHI00315,a3a547b8-a135-11e9-9cda-0a87ae2ba916,"{'android': 'https://chi.lft.to/lastmile_qr_scan', 'ios': 'https://chi.lft.to/lastmile_qr_scan'}",NaN,NaN,1379
876,900 W Harrison St,19,-87.649807,41.874754,CHI01748,a3a56ae5-a135-11e9-9cda-0a87ae2ba916,"{'android': 'https://chi.lft.to/lastmile_qr_scan', 'ios': 'https://chi.lft.to/lastmile_qr_scan'}",NaN,NaN,1385
243,Aberdeen St & 103rd St,16,-87.650220,41.707040,CHI01976,2014856134756363200,"{'android': 'https://chi.lft.to/lastmile_qr_scan', 'ios': 'https://chi.lft.to/lastmile_qr_scan'}",NaN,NaN,1095


In [59]:
# Variables de entrada
X = df_model.drop(columns=['start_station_idx'])
y_start = df_model['start_station_idx']
y_end   = df_model['end_station_idx']

In [60]:
len(y_start)

9510782

In [61]:
result = df_model['start_station_idx'].isin(df_bikesStations['station_idx'])

In [62]:
len(result)

9510782

## Como todas las estaciones estan presentes se crea el one-hotEncoding

In [65]:
from sklearn.model_selection import train_test_split
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

X_train, X_test, y_start_train, y_start_test, y_end_train, y_end_test = train_test_split(
    X, 
    y_start, 
    y_end, 
    test_size=0.2, 
    random_state=42
)

# ---------- Normalizar índices de estaciones con un offset común
station_offset = int(min(y_start.min(), y_end.min()))
# desplazamos todos para que el mínimo sea 0
y_start_train_shift = (y_start_train - station_offset).astype(int)
y_start_test_shift  = (y_start_test  - station_offset).astype(int)
y_end_train_shift   = (y_end_train   - station_offset).astype(int)
y_end_test_shift    = (y_end_test    - station_offset).astype(int)

num_stations = int(max(y_start.max(), y_end.max()) - station_offset + 1)
num_features = X.shape[1]

print(f"Number of stations: {num_stations}")
print(f"Number of features: {num_features}")

# ---------- Preparar inputs en la forma que requieren los Embeddings: (n,1)
X_train_context = X_train.values.astype(np.float32)
X_test_context  = X_test.values.astype(np.float32)

start_train_input = y_start_train_shift.values.reshape(-1, 1)
start_test_input  = y_start_test_shift.values.reshape(-1, 1)

# Labels (etiquetas) para clasificación: estación destino (y_end)
y_train_labels = y_end_train_shift.values.astype(int)
y_test_labels  = y_end_test_shift.values.astype(int)

# ---------- Opcional: split del train en train/val
X_t_ctx, X_val_ctx, start_t_in, start_val_in, y_t, y_val = train_test_split(
    X_train_context,
    start_train_input,
    y_train_labels,
    test_size=0.2,
    random_state=42
)

# ---------- Construcción del modelo (inputs y embeddings)
embedding_dim = int(np.ceil(np.sqrt(num_stations)))

context_input = layers.Input(shape=(num_features,), name="context")
start_input   = layers.Input(shape=(1,), name="start_station_input")
#end_input     = layers.Input(shape=(1,), name="end_station_input")

start_embed = layers.Embedding(input_dim=num_stations, output_dim=embedding_dim, name='start_embedding')(start_input)
start_embed = layers.Flatten()(start_embed)

# end_embed = layers.Embedding(input_dim=num_stations, output_dim=embedding_dim, name='end_embedding')(end_input)
# end_embed = layers.Flatten()(end_embed)

x = layers.Concatenate()([context_input, start_embed])
x = layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)

end_output = layers.Dense(num_stations, activation='softmax', name='end_station')(x)

model = Model(inputs=[context_input, start_input], outputs=end_output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Number of stations: 1912
Number of features: 28


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ start_station_input │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ start_embedding     │ (None, 1, 44)     │     84,128 │ start_station_in… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context             │ (None, 28)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 44)        │          0 │ start_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 72)        │          0 │ context[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │      9,344 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ end_station (Dense) │ (None, 1912)      │    124,280 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 226,008 (882.84 KB)

 Trainable params: 226,008 (882.84 KB)

 Non-trainable params: 0 (0.00 B)

In [66]:
%%time

# ---------- Callbacks
callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# ---------- Entrenamiento usando validation_data (más explícito y claro)
history = model.fit(
    {'context': X_t_ctx, 'start_station_input': start_t_in},
    y_t,
    validation_data=(
        {'context': X_val_ctx, 'start_station_input': start_val_in},
        y_val
    ),
    epochs=20,
    batch_size=128,
    callbacks=[callback]
)

Epoch 1/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 155s 3ms/step - accuracy: 0.0595 - loss: 4.7365 - val_accuracy: 0.1184 - val_loss: 3.5919
Epoch 2/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 156s 3ms/step - accuracy: 0.1149 - loss: 3.6309 - val_accuracy: 0.1537 - val_loss: 3.3004
Epoch 3/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 149s 3ms/step - accuracy: 0.1345 - loss: 3.4784 - val_accuracy: 0.1287 - val_loss: 3.3304
Epoch 4/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 158s 3ms/step - accuracy: 0.1449 - loss: 3.3877 - val_accuracy: 0.1654 - val_loss: 3.2450
Epoch 5/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 222s 4ms/step - accuracy: 0.1584 - loss: 3.3008 - val_accuracy: 0.1490 - val_loss: 3.1389
Epoch 6/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 179s 4ms/step - accuracy: 0.1681 - loss: 3.2340 - val_accuracy: 0.1715 - val_loss: 3.1017
Epoch 7/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 178s 4ms/step - accuracy: 0.1775 - loss: 3.1713 - val_accuracy: 0.1516 - val_loss: 3.2255
Epoch 8/20
47554/47554 ━━━━━━━━━━━━━━━━━━━━ 177s 4ms/step - ac